<div style="background: linear-gradient(135deg, #0a0a2e 0%, #1a1a5e 50%, #1a759f 100%); padding: 35px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; font-family: 'Segoe UI', sans-serif; font-size: 2.2em; margin: 0; font-weight: 700;">🔍 Interpretabilidad con SHAP</h1>
  <p style="color: #a8d8ea; font-size: 1.1em; margin: 12px 0 0;">E-Commerce Churn Prediction &nbsp;|&nbsp; No Country — Equipo 40</p>
</div>

> **¿Por qué SHAP?** Los modelos de boosting y random forest son cajas negras.  
> SHAP (SHapley Additive exPlanations) asigna a **cada feature** su contribución exacta en cada predicción individual.  
> Esto permite responder: *'¿Por qué el modelo dice que este cliente tiene 87% de probabilidad de irse?'*

---

## 📋 Tabla de Contenidos

| # | Sección | Descripción |
|---|---------|-------------|
| 1 | [⚙️ Configuración](#1) | Setup y carga del modelo champion |
| 2 | [🧮 Calcular Valores SHAP](#2) | TreeExplainer para RF/XGBoost |
| 3 | [🐝 Summary Plot (Beeswarm)](#3) | Impacto global de cada feature |
| 4 | [📊 Bar Chart de Importancia](#4) | Ranking de features por importancia media |
| 5 | [📉 Dependence Plots](#5) | Relación feature-valor vs SHAP |
| 6 | [💧 Waterfall — Cliente Individual](#6) | Explicación por cliente |
| 7 | [✅ Conclusiones](#7) | Insights para el equipo de negocio |

---
## ⚙️ 1. Configuración <a id='1'></a>

> Se carga el modelo **champion** del registry y los datos procesados.  
> Si SHAP no está instalado, se provee el comando de instalación.

In [ ]:
import sys, os
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
os.environ['DISABLE_PANDERA_IMPORT_WARNING'] = 'True'

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

try:
    import shap
    print(f'✅ SHAP {shap.__version__} disponible')
except ImportError:
    print('⚠️  SHAP no instalado. Ejecuta: pip install shap')
    shap = None

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'primary':'#2d6a4f','secondary':'#52b788','accent':'#1a759f','warning':'#e76f51','danger':'#d62828'}

with open('config/config.yaml') as f:
    cfg = yaml.safe_load(f)

Path('reports/figures/shap').mkdir(parents=True, exist_ok=True)
print('✅ Entorno configurado')

In [ ]:
from src.modeling.registry import ModelRegistry
from src.features.scaler import FeatureScaler

# Cargar modelo champion
registry = ModelRegistry()
model, champion_info = registry.load_champion()
FEATURES = champion_info.get('features', cfg['features']['rfm'])
MODEL_NAME = champion_info.get('model_name', 'Champion')

print(f'🏆 Modelo champion cargado : {MODEL_NAME}')
print(f'📋 Features del modelo     : {FEATURES}')

# Cargar datos
rfm = pd.read_parquet('data/processed/rfm_with_churn.parquet')
features_available = [f for f in FEATURES if f in rfm.columns]

scaler_path = Path('models/champion/scaler.pkl')
if scaler_path.exists():
    scaler = FeatureScaler.load(scaler_path, features=features_available)
    rfm_scaled = scaler.transform(rfm)
else:
    rfm_scaled = rfm.copy()

X = rfm_scaled[features_available]
print(f'\n✅ Dataset listo: {X.shape[0]:,} clientes × {X.shape[1]} features')

---
## 🧮 2. Calcular Valores SHAP <a id='2'></a>

> El `ModelInterpreter` selecciona automáticamente el tipo de Explainer según el modelo:
>
> | Modelo | Explainer SHAP | Complejidad |
> |--------|---------------|-------------|
> | Random Forest | `TreeExplainer` | O(TLD) — muy rápido |
> | XGBoost | `TreeExplainer` | O(TLD) — muy rápido |
> | Logistic Regression | `LinearExplainer` | O(N×F) — rápido |
> | Otros | `KernelExplainer` | O(N²) — lento, requiere sample |

In [ ]:
from src.modeling.interpreter import ModelInterpreter

interpreter = ModelInterpreter(
    figures_dir='reports/figures/shap',
    sample_size=cfg['interpretability']['shap']['sample_size'],
    max_display=cfg['interpretability']['shap']['max_display']
)

print(f'⏳ Calculando valores SHAP (muestra: {cfg["interpretability"]["shap"]["sample_size"]} clientes)...')
interpreter.explain(model, MODEL_NAME, X)
print('✅ Valores SHAP calculados correctamente.')

---
## 🐝 3. SHAP Summary Plot (Beeswarm) <a id='3'></a>

> **Cómo leer este gráfico:**
> - **Eje X:** Valor SHAP (impacto en la predicción). Positivo → aumenta probabilidad de Churn.
> - **Color:** Valor real de la feature. Rojo = alto, azul = bajo.
> - **Ancho del enjambre:** Densidad de clientes con ese valor SHAP.

In [ ]:
print('📊 Generando SHAP Summary Plot (Beeswarm)...')
summary_path = interpreter.plot_summary()
print(f'✅ Guardado en: {summary_path}')

---
## 📊 4. SHAP Feature Importance (Bar Chart) <a id='4'></a>

> La **importancia media |SHAP|** promedia el impacto absoluto de cada feature en todas las predicciones.  
> Es más informativa que la importancia de Random Forest (basada en impureza de Gini),  
> ya que considera tanto las predicciones de churn como de no-churn.

In [ ]:
bar_path = interpreter.plot_bar_importance()
print(f'✅ Bar chart guardado en: {bar_path}')
print()

importance_df = interpreter.get_feature_importance_df()
print('=== Importancia de Features por SHAP (Mean |SHAP value|) ===')
for i, row in importance_df.iterrows():
    bar_vis = '█' * int(row['mean_abs_shap'] / importance_df['mean_abs_shap'].max() * 25)
    print(f'  #{i+1:02d}  {row["feature"]:<25} {bar_vis:<25} {row["mean_abs_shap"]:.5f}')

---
## 📉 5. Dependence Plots — Top 3 Features <a id='5'></a>

> Los **Dependence Plots** muestran cómo varía el SHAP value a medida que cambia el valor de una feature,  
> con el color indicando una segunda feature de interacción detectada automáticamente por SHAP.

> **Interpretación:**  
> Si `Recency` tiene SHAP positivo cuando su valor es alto → confirma que alta inactividad → mayor riesgo de churn.

In [ ]:
top_features = importance_df['feature'].head(3).tolist()
print(f'📉 Generando Dependence Plots para top 3: {top_features}')

for feature in top_features:
    try:
        dep_path = interpreter.plot_dependence(feature)
        print(f'  ✅ {feature} → {dep_path}')
    except Exception as e:
        print(f'  ⚠️  {feature}: {e}')

---
## 💧 6. Waterfall — Explicación de Clientes Individuales <a id='6'></a>

> El **Waterfall Plot** es la forma más intuitiva de explicar una predicción individual a un stakeholder:  
> *'Este cliente tiene 91% de probabilidad de churn PORQUE su Recency es +45 días sobre el promedio'*

> | Elemento | Significado |
> |----------|-------------|
> | `E[f(X)]` | Valor base del modelo (predicción para un cliente promedio) |
> | Barras rojas | Features que **aumentan** la probabilidad de churn |
> | Barras azules | Features que **reducen** la probabilidad de churn |
> | `f(X)` | Predicción final para este cliente |

In [ ]:
if shap is not None:
    rfm_with_prob = rfm_scaled.copy()
    rfm_with_prob['ChurnProb'] = model.predict_proba(X)[:, 1]
    rfm_with_prob['CHURN'] = rfm['CHURN'].values

    high_risk_idx = rfm_with_prob['ChurnProb'].idxmax()
    low_risk_idx  = rfm_with_prob['ChurnProb'].idxmin()

    print('=== Caso #1: Cliente de MÁXIMO riesgo ===')
    print(rfm_with_prob.loc[high_risk_idx, features_available + ['ChurnProb']].round(4))

    print('\n=== Caso #2: Cliente de MÍNIMO riesgo ===')
    print(rfm_with_prob.loc[low_risk_idx, features_available + ['ChurnProb']].round(4))
else:
    print('⚠️  SHAP no instalado. Instala con: pip install shap')

In [ ]:
if shap is not None and MODEL_NAME in ('RandomForest', 'XGBoost'):
    explainer = shap.TreeExplainer(model)
    sample_case = X.loc[[high_risk_idx]]
    shap_vals = explainer.shap_values(sample_case)

    sv = shap_vals[1][0] if isinstance(shap_vals, list) else shap_vals[0]
    base = (explainer.expected_value[1]
            if isinstance(explainer.expected_value, (list, np.ndarray))
            else explainer.expected_value)

    shap_exp = shap.Explanation(
        values=sv, base_values=base,
        data=sample_case.values[0],
        feature_names=features_available
    )

    plt.figure(figsize=(11, 5))
    shap.plots.waterfall(shap_exp, show=False)
    plt.title(f'💧 SHAP Waterfall — Cliente de Mayor Riesgo\nProb={rfm_with_prob.loc[high_risk_idx, "ChurnProb"]:.3f}',
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('reports/figures/shap/shap_waterfall_high_risk.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Guardado: reports/figures/shap/shap_waterfall_high_risk.png')
else:
    print('⚠️  Waterfall disponible solo para RF/XGBoost con SHAP instalado.')

---
## ✅ 7. Conclusiones — Insights para Negocio <a id='7'></a>

In [ ]:
top3 = importance_df['feature'].head(3).tolist()
print('╔══════════════════════════════════════════════════════╗')
print('║    🔍 INSIGHTS DE INTERPRETABILIDAD (SHAP)          ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'  Modelo explicado : {MODEL_NAME}')
print(f'  Top 3 features   : {top3}')
print('╠══════════════════════════════════════════════════════╣')
for i, row in importance_df.iterrows():
    bar = '█' * int(row['mean_abs_shap'] / importance_df['mean_abs_shap'].max() * 20)
    print(f'  #{i+1:02d} {row["feature"]:<22} {bar:<20} {row["mean_abs_shap"]:.5f}')
print('╠══════════════════════════════════════════════════════╣')
print(f'  💡 {top3[0]} es la variable más determinante del churn')
print(f'  💡 Clientes con alta {top3[0]} tienen mayor riesgo')
print('  📁 Gráficos en: reports/figures/shap/')
print('  → Siguiente: 06_segmentation_analysis.ipynb')
print('╚══════════════════════════════════════════════════════╝')

<div style="background: #e8f0fe; border-left: 5px solid #1a1a5e; padding: 15px 20px; border-radius: 6px; margin-top: 20px;">
  <strong>✅ Interpretabilidad completada.</strong> Ahora sabemos <em>qué</em> impulsa las predicciones del modelo.<br><br>
  <strong>Próxima acción para el equipo de negocio:</strong> El factor más importante es <code>Recency</code>:<br>
  diseñar campañas que se activen cuando un cliente no compra en más de <strong>45-60 días</strong> (antes del umbral de churn).
  <br><br>Continúa en <strong>06_segmentation_analysis.ipynb</strong>.
</div>